In [11]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../')
from utils.MultiLabelPredictor import MultilabelPredictor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
def best_for_each(r2s,mses):
    best_r2_idxs = np.argmax(r2s, axis=0)
    best_mse_idxs = np.argmin(mses, axis=0)
    best_r2_idx = Counter(best_r2_idxs).most_common(1)[0][0] 
    best_mse_idx = Counter(best_mse_idxs).most_common(1)[0][0] 
    return best_r2_idx == best_mse_idx, best_mse_idx 

In [13]:
class PredictExposure:
    def __init__(self, model):
        self.model = model 

    def fit(self, mutation_count):

        mutation_count_bin = pd.read_csv('../simulations/ground_truth/bin_exposures.csv').iloc[:,1:].astype(int).values  # N x 29
        signature_exposure = pd.read_csv('../simulations/ground_truth/exposures.csv').iloc[:,1:].values  # N x 29

        X_train, X_test, y_bin_train, y_bin_test, y_exp_train, y_exp_test = train_test_split(
            mutation_count, mutation_count_bin, signature_exposure, train_size=0.8,
        )
        regressors = []
        r2_scores = []
        mse_scores = []
        predictions = []

        for i in range(y_bin_train.shape[1]):  # ciclo sulle 29 firme           # Fa predizioni anche su segnature che nono dovrebbero esserci capire come mais
            # Filtra campioni con firma attiva in train
            idx_train = y_bin_train[:, i] == 1
            pdf(idx_train)
            X_train_i = X_train[idx_train]
            pdf(X_train_i)
            y_train_i = y_exp_train[idx_train, i]
            pdf(y_train_i)

            # Filtra campioni con firma attiva in test
            idx_test = y_bin_test[:, i] == 1
            X_test_i = X_test[idx_test]
            y_test_i = y_exp_test[idx_test, i]

            if len(y_train_i) == 0 or len(y_test_i) == 0:
                regressors.append(None)
                r2_scores.append(np.nan)
                mse_scores.append(np.nan)
                continue

            self.model.fit(X_train_i, y_train_i)
            regressors.append(self.model)
            y_pred_i = self.predict(X_test_i)
            predictions.append(y_pred_i)
            r2_scores.append(r2_score(y_test_i, y_pred_i))
            mse_scores.append(mean_squared_error(y_test_i, y_pred_i))
            
        # # Output risultati
        # for i in range(len(regressors)):
        #     print(f"Signature_{i+1}: R2 = {r2_scores[i]:.3f}, MSE = {mse_scores[i]:.3e}")
        return r2_scores, mse_scores, predictions

    def predict(self, X_test_i):
            y_pred_i = self.model.predict(X_test_i)
            return y_pred_i

In [2]:
class PredictExposure:
    def __init__(self, model):
        self.model = model
        self.regressors = []
        self.mutation_count_bin = pd.read_csv('../simulations/ground_truth/bin_exposures.csv').iloc[:,1:].astype(int).values  # N x 29
        self.signature_exposure = pd.read_csv('../simulations/ground_truth/exposures.csv').iloc[:,1:].values  # N x 29


    def fit(self, train_data):
        for i in range(self.mutation_count_bin.shape[1]):  # ciclo sulle 29 firme
            # Filtra campioni con firma attiva in train
            idx_train = self.mutation_count_bin[:, i] == 1
            X_train_i = train_data[idx_train]
            y_train_i = self.signature_exposure[idx_train, i]

            self.model.fit(X_train_i, y_train_i)
            self.regressors.append(self.model)
            
    def evaluate(self, evaluate):
        X_train, X_test, y_bin_train, y_bin_test, y_exp_train, y_exp_test = train_test_split(
            evaluate, self.mutation_count_bin, self.signature_exposure, train_size=0.8,
        )
        r2_scores = []
        mse_scores = []
        predictions = []

        for i in range(y_bin_train.shape[1]):  # ciclo sulle 29 firme
            # Filtra campioni con firma attiva in train
            idx_train = y_bin_train[:, i] == 1
            X_train_i = X_train[idx_train]
            y_train_i = y_exp_train[idx_train, i]

            # Filtra campioni con firma attiva in test
            idx_test = y_bin_test[:, i] == 1
            X_test_i = X_test[idx_test]
            y_test_i = y_exp_test[idx_test, i]

            if len(y_train_i) == 0 or len(y_test_i) == 0:
                self.regressors.append(None)
                r2_scores.append(np.nan)
                mse_scores.append(np.nan)
                continue

            regressor = self.model.fit(X_train_i, y_train_i)
            self.regressors.append(regressor)
            y_pred_i = regressor.predict(X_test_i)
            predictions.append(y_pred_i)
            r2_scores.append(r2_score(y_test_i, y_pred_i))
            mse_scores.append(mean_squared_error(y_test_i, y_pred_i))
            
        # # Output risultati
        # for i in range(len(regressors)):
        #     print(f"Signature_{i+1}: R2 = {r2_scores[i]:.3f}, MSE = {mse_scores[i]:.3e}")
        return r2_scores, mse_scores, predictions

    def predict(self, test_data):
        y_pred_i = []
        for i in range(self.mutation_count_bin.shape[1]):
            idx_test = self.mutation_count_bin[:, i] == 1
            X_test_i = test_data[idx_test]
            y_test_i = self.signature_exposure[idx_test, i]
            y_pred_i.append(self.regressors[i].predict(X_test_i))
        return y_pred_i
            

In [6]:
predictor = MultilabelPredictor.load('../models/save/Predictior-0.03')
pred_mutation_count = pd.read_csv('../simulations/data2/run_1/trinucleotides_counts_sampling_0.03.csv').iloc[:,1:]
prediction = predictor.predict(pred_mutation_count)

Predicting with TabularPredictor for label: S1 (SBS1 - 0.99) ...
Predicting with TabularPredictor for label: S2 (SBS2 - 0.99) ...
Predicting with TabularPredictor for label: S3 (SBS3 - 0.97) ...
Predicting with TabularPredictor for label: S4 (SBS4 - 0.98) ...
Predicting with TabularPredictor for label: S5 (SBS5 - 0.98) ...
Predicting with TabularPredictor for label: S6 (SBS7a - 1.00) ...
Predicting with TabularPredictor for label: S7 (SBS7b - 0.96) ...
Predicting with TabularPredictor for label: S8 (SBS8 - 0.92) ...
Predicting with TabularPredictor for label: S9 (SBS9 - 0.94) ...
Predicting with TabularPredictor for label: S10 (SBS10a - 1.00) ...
Predicting with TabularPredictor for label: S11 (SBS10d - 0.98) ...
Predicting with TabularPredictor for label: S12 (SBS11 - 0.99) ...
Predicting with TabularPredictor for label: S13 (SBS13 - 0.99) ...
Predicting with TabularPredictor for label: S14 (SBS14 - 0.98) ...
Predicting with TabularPredictor for label: S15 (SBS15 - 0.97) ...
Predictin

### Train's Data

In [4]:
seed = np.random.randint(1,100000)
np.random.seed(seed=seed)
# Loading the data
signature_prob_distribution = pd.read_csv('../simulations/ground_truth/signatures.csv').iloc[:,1:].values
mutation_count = pd.read_csv('../simulations/data/run_1/trinucleotides_counts_sampling_0.03.csv').iloc[:,1:].values  # N x 96
mutation_count_bin = pd.read_csv('../simulations/ground_truth/bin_exposures.csv').iloc[:,1:].astype(int).values  # N x 29
tissues = pd.read_csv('../simulations/ground_truth/tumor_site.csv').iloc[:,1:-1].values

In [5]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
encoded_labels = pd.DataFrame(encoder.fit_transform(tissues))

/home/massimo/Documents/stage/signature_inference/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:114: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [10]:
with_bin_mutation_count = np.hstack([mutation_count @ signature_prob_distribution.T, mutation_count_bin])
pe = PredictExposure(LinearRegression())
r2, mse, pred = pe.fit(with_bin_mutation_count)
df = pd.DataFrame(pred)
df
# print(f'Mean: {np.mean(r2, axis=0)}, Median {np.median(r2, axis=0)}, Max/Min: {np.max(r2, axis=0)}/{np.min(r2, axis=0)}')  #83637

,0,1,2,3,4,5,6,7,8,9,...,3496,3497,3498,3499,3500,3501,3502,3503,3504,3505
0,1615.956541,1042.278025,4862.243041,2955.675466,2420.078576,1412.277187,782.828237,838.711481,1984.740185,771.252535,...,-377.313911,1644.668862,6169.487612,72.071094,348.978683,1033.934932,811.371244,1103.693172,405.970533,2061.1396
1,2552.857918,135.959462,178.556595,164.603737,293.552388,4.659915,91.444711,-40.476631,3055.630828,630.796840,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,6814.606337,171.178502,1202.941046,55.232033,247.099000,1630.510521,251.364718,1249.351319,3166.292944,3245.165556,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,16714.583706,-375.608729,1709.685824,-463.409015,901.833104,-121.581655,290.718273,-563.457393,-364.916828,203.088626,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2589.963508,-193.339224,840.383140,215.068410,581.686362,290.466428,870.139667,232.569154,1168.535878,396.740480,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1116.261450,873.754737,-53.794785,1203.067081,1043.931692,229.875343,-464.780688,92.298131,432.861440,515.819165,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,-698.295633,-2.608313,527.617644,-478.900629,-298.259110,439.699477,1042.155334,-71.760425,-230.735829,253.893644,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,3811.041700,1074.128629,1029.265868,-1.439425,208.534407,1742.498014,783.761706,138.181615,227.427112,1348.023686,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2112.372249,396.527435,579.498238,1808.991846,3411.360014,3696.133224,381.346381,-254.463728,-123.911455,918.240939,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,681.411108,-138.178144,466.234872,140.183511,-203.442973,4.543769,-1392.427313,-1845.911419,223.476386,813.767721,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
pred_with_bin_mutation_count = np.hstack([mutation_count @ signature_prob_distribution.T, prediction])
pred = pe.predict(pred_with_bin_mutation_count)
df = pd.DataFrame(pred)
df
# print(f'Mean: {np.mean(pred_r2, axis=0)}, Median {np.median(pred_r2, axis=0)}, Max/Min: {np.max(pred_r2, axis=0)}/{np.min(pred_r2, axis=0)}')  #83637

,0,1,2,3,4,5,6,7,8,9,...,17443,17444,17445,17446,17447,17448,17449,17450,17451,17452
0,353.333076,-233.917950,-42.477325,-233.489408,-313.596247,175.722660,-88.937408,-545.893649,120.778267,363.474986,...,-139.879707,-12.610569,428.995971,283.155572,-84.142385,22.81961,3.18985,-219.215379,155.389423,-54.439599
1,353.333076,-42.477325,-233.489408,175.722660,-88.937408,-545.893649,120.778267,363.474986,-139.864350,-270.428297,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,353.333076,-233.489408,175.722660,120.778267,-139.864350,-270.428297,400.793524,-247.302824,-146.485060,4.945307,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,353.333076,-42.477325,-233.489408,175.722660,120.778267,363.474986,-139.864350,400.793524,493.710715,-146.485060,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,353.333076,-233.917950,-42.477325,-233.489408,-313.596247,175.722660,-88.937408,-545.893649,363.474986,-139.864350,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,353.333076,-42.477325,-233.489408,-313.596247,175.722660,-88.937408,3433.600763,-545.893649,120.778267,363.474986,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,-42.477325,-233.489408,120.778267,363.474986,-139.864350,-270.428297,400.793524,493.710715,-146.485060,4.945307,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,353.333076,-233.489408,175.722660,363.474986,-139.864350,-270.428297,400.793524,-146.485060,4.945307,281.294121,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,175.722660,-545.893649,120.778267,-139.864350,-270.428297,-247.302824,-146.485060,-488.482879,513.199073,314.108490,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,3433.600763,-270.428297,-146.485060,227.520200,89.042498,-154.968001,193.368054,-153.293407,-224.725333,134.741289,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
seed

83637

In [8]:
scaler = StandardScaler()
mutation_scaled = scaler.fit_transform(mutation_count)
with_bin_mutation_count = np.hstack([mutation_count @ signature_prob_distribution.T, mutation_count_bin])
with_bin_sites_mutation_count = np.hstack([mutation_count @ signature_prob_distribution.T, mutation_count_bin,encoded_labels])

models = [LinearRegression(),
          LinearRegression(),
          LinearRegression(),
          LinearRegression(),       # BEST
          Ridge(alpha=0.1),
          LinearRegression()
          ]
fit_inputs = [mutation_count,                                   # Simple count of mutation per sample
              mutation_count @ signature_prob_distribution.T,   # Count of mutation per sample multiplied by the mutation distribution
              mutation_scaled @ signature_prob_distribution.T,  # Count of mutation per sample normalized multiplied by the mutation distribution
              with_bin_mutation_count, # BEST                   # Count multiplied by probability distribution concat with the True/Fasle vector
              mutation_count @ signature_prob_distribution.T,   # Count of mutation per sample multiplied by the mutation distribution
              with_bin_sites_mutation_count
              ]

bests = []
best_r2s = []
best_mses = []
i = 0
while i in range(50):
    i+=1
    r2s = []
    mses = []
    print(f'fitting {i}')
    for model, fit_input in zip(models, fit_inputs):
        pe = PredictExposure(model)
        fit_r2, fit_mse = pe.fit(fit_input)
        r2s.append(fit_r2),
        mses.append(fit_mse)
    match, best = best_for_each(r2s,mses)
    if match:
        print(f'Match on {best}')
        print(r2s[best], mses[best])
        print(best)
        bests.append(best)
        best_mses.append(mses[best])
        best_r2s.append(r2s[best])

fitting 1


KeyboardInterrupt: 

In [ ]:
Counter(bests).most_common(1)[0][0] 


5

In [ ]:
df = pd.concat([pd.DataFrame(bests, columns=['best']), pd.DataFrame(best_r2s)], axis=1)
df[df['best'] == 5].iloc[:, 1:]

NameError: name 'pd' is not defined